# 02 · Day-ahead load forecast
**Forecast definition:** predict each hour's ERCOT demand one day ahead. Every feature must be known a day earlier: calendar fields, demand lags of 24 h and 168 h, and temperature.
**Limitation:** temperature is *observed* temperature standing in for a day-ahead weather forecast, so real-world error would be somewhat higher.
**Split:** train 2019–2023, test 2024–2025 (by time, never random).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()

In [2]:
df = q("SELECT * FROM model_hourly ORDER BY period")
df = df.iloc[168:].copy()          # first 168 rows have no lag_168
df["cooling_degree"] = (df.avg_temp_c - 18.3).clip(lower=0)
df["heating_degree"] = (18.3 - df.avg_temp_c).clip(lower=0)
df["temp_sq"] = df.avg_temp_c ** 2
df["temp_delta_24"] = df.avg_temp_c - df.temp_lag_24
df["is_weekend"] = df.is_weekend.astype(int)
df["is_holiday"] = df.is_holiday.astype(int)
print(len(df), "rows;", df[["demand_mwh", "demand_lag_24", "demand_lag_168", "avg_temp_c"]].isna().sum().sum(), "nulls in key columns")
train = df[df.period.dt.year <= 2023].copy()
test = df[df.period.dt.year >= 2024].copy()
print(f"train {len(train):,} rows ({train.period.min():%Y-%m-%d} to {train.period.max():%Y-%m-%d})")
print(f"test  {len(test):,} rows ({test.period.min():%Y-%m-%d} to {test.period.max():%Y-%m-%d})")

61200 rows; 0 nulls in key columns
train 43,656 rows (2019-01-08 to 2023-12-31)
test  17,544 rows (2024-01-01 to 2025-12-31)


/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_24538/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


## Features

In [3]:
TEMP = ["avg_temp_c", "cooling_degree", "heating_degree", "temp_sq"]
FLAGS = ["is_weekend", "is_holiday"]
LAGS = ["demand_lag_24", "demand_lag_168"]
CAL = ["local_hour", "day_of_week", "month"]
BASE_FEATURES = TEMP + FLAGS + LAGS + CAL
FINAL_FEATURES = BASE_FEATURES + ["temp_delta_24"]

## Models

In [4]:
import statsmodels.api as sm
from sklearn.ensemble import HistGradientBoostingRegressor

def metrics(y, p):
    e = y - p
    return {"RMSE (MWh)": np.sqrt((e ** 2).mean()), "MAPE (%)": 100 * (e.abs() / y).mean()}

preds = {}
# 1. Baseline: same hour one week earlier
preds["Baseline: same hour last week"] = test.demand_lag_168
# Extra reference: same hour yesterday
preds["Reference: same hour yesterday"] = test.demand_lag_24

# 2. Linear regression. Calendar fields as dummies. avg_temp_c is dropped because
#    cooling_degree - heating_degree = avg_temp_c - 18.3 (perfect collinearity), and
#    is_weekend is dropped because the day_of_week dummies already encode it.
OLS_NUM = ["cooling_degree", "heating_degree", "temp_sq", "temp_delta_24", "is_holiday"] + LAGS
def design(d, cols=None):
    X = pd.concat([d[OLS_NUM], pd.get_dummies(d[CAL].astype(str), drop_first=True, dtype=float)], axis=1)
    X = sm.add_constant(X, has_constant="add")
    return X if cols is None else X.reindex(columns=cols, fill_value=0)
train_ols = train.dropna(subset=["temp_delta_24"])
X_train = design(train_ols)
ols = sm.OLS(train_ols.demand_mwh, X_train).fit()
preds["Linear regression (OLS)"] = ols.predict(design(test, X_train.columns))
print(f"OLS R^2 on train: {ols.rsquared:.3f}")

# 3a. Gradient boosting, default settings, predicting load directly
gb_default = HistGradientBoostingRegressor(random_state=0).fit(train[BASE_FEATURES], train.demand_mwh)
preds["Gradient boosting (default)"] = gb_default.predict(test[BASE_FEATURES])

# 3b. Gradient boosting, default settings, predicting the change from the same hour yesterday.
#     Trees cannot extrapolate past the training range; anchoring on yesterday's load lets the
#     model handle 2024-25 load levels it never saw in training.
gb = HistGradientBoostingRegressor(random_state=0).fit(
    train_ols[FINAL_FEATURES], train_ols.demand_mwh - train_ols.demand_lag_24)
preds["Gradient boosting (change vs yesterday)"] = test.demand_lag_24 + gb.predict(test[FINAL_FEATURES])

results = pd.DataFrame({k: metrics(test.demand_mwh, v) for k, v in preds.items()}).T.round(2)
results["RMSE (MWh)"] = results["RMSE (MWh)"].round(0).astype(int)
results

OLS R^2 on train: 0.946


,RMSE (MWh),MAPE (%)
Baseline: same hour last week,6480,8.53
Reference: same hour yesterday,3503,4.52
Linear regression (OLS),3623,5.55
Gradient boosting (default),3834,5.77
Gradient boosting (change vs yesterday),2187,2.94


In [5]:
best_name = results["MAPE (%)"].idxmin()
print("best model:", best_name)
test["pred_best"] = preds[best_name]
test["pred_baseline"] = preds["Baseline: same hour last week"]
test["ape_best"] = 100 * (test.demand_mwh - test.pred_best).abs() / test.demand_mwh
test["ape_baseline"] = 100 * (test.demand_mwh - test.pred_baseline).abs() / test.demand_mwh
results.to_csv("data/clean/model_results.csv")
test[["period", "local_ts", "local_hour", "season", "is_holiday", "demand_mwh", "avg_temp_c",
      "pred_best", "pred_baseline", "ape_best", "ape_baseline"]].to_parquet("data/clean/test_predictions.parquet", index=False)

best model: Gradient boosting (change vs yesterday)


## Where does it fail?

In [6]:
by_hour = test.groupby("local_hour")[["ape_best", "ape_baseline"]].mean().round(2)
by_season = test.groupby("season")[["ape_best", "ape_baseline"]].mean().round(2).loc[["Winter", "Spring", "Summer", "Fall"]]
display(by_season); by_hour.T

,ape_best,ape_baseline
season,,
Winter,3.68,12.10
Spring,3.02,7.49
Summer,2.47,6.70
Fall,2.58,7.87


local_hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
ape_best,2.90,2.69,2.62,2.51,2.47,2.53,2.72,2.85,2.83,2.63,2.78,2.83,2.92,2.95,3.03,3.06,3.15,3.27,3.44,3.24,3.37,3.37,3.30,2.97
ape_baseline,8.46,8.53,8.61,8.63,8.57,8.58,8.50,8.37,8.23,7.95,7.80,7.87,8.17,8.60,9.04,9.28,9.25,9.03,8.77,8.65,8.58,8.45,8.39,8.38


In [7]:
print("holiday vs non-holiday MAPE (best model):")
display(test.groupby("is_holiday").ape_best.mean().round(2))
worst_days = (test.groupby(test.local_ts.dt.date)
                  .agg(mape=("ape_best", "mean"), peak_mwh=("demand_mwh", "max"),
                       min_temp=("avg_temp_c", "min"), max_temp=("avg_temp_c", "max"), holiday=("is_holiday", "max"))
                  .sort_values("mape", ascending=False).head(10).round(1))
worst_days

holiday vs non-holiday MAPE (best model):


is_holiday
0    2.90
1    4.05
Name: ape_best, dtype: float64

,mape,peak_mwh,min_temp,max_temp,holiday
local_ts,,,,,
2024-07-08,17.8,55545.0,25.1,30.6,0
2024-01-15,15.6,76339.0,-6.4,1.8,1
2025-01-20,12.8,73937.0,-4.0,2.4,1
2025-01-02,9.4,51821.0,7.2,15.9,0
2025-04-05,9.2,50935.0,12.1,20.2,0
2024-01-16,9.1,78155.0,-8.4,-0.2,0
2025-11-29,9.0,50988.0,14.6,23.0,0
2025-03-08,8.7,52572.0,11.1,24.2,0
2024-01-18,8.4,60103.0,3.2,18.0,0


In [8]:
import plotly.graph_objects as go
from viz import add_caption, save, SERIES
fig = go.Figure()
for i, (col, name) in enumerate([("ape_baseline", "Baseline (last week)"), ("ape_best", best_name)]):
    fig.add_bar(x=by_season.index, y=by_season[col], name=name, marker_color=SERIES[i])
fig.update_layout(title="Forecast error by season, test set 2024–2025", barmode="group", bargap=0.35, bargroupgap=0.08,
                  yaxis_title="Mean absolute % error", xaxis_title="Season")
add_caption(fig, f"Error is highest in winter for both models: cold snaps (and holidays such as MLK Day during them) are the hardest days to forecast.")
save(fig, "model_error_by_season")